# 02 - Training Windows

Create input and target sequences for next-token prediction. Every target is its corresponding input shifted one token to the left.

In [ ]:
from pathlib import Path
import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root')


raw_text = (find_repo_root() / 'data' / 'the-verdict.txt').read_text(encoding='utf-8')
tokenizer = tiktoken.get_encoding('gpt2')
token_ids = tokenizer.encode(raw_text)
print(f'{len(token_ids):,} token IDs')

In [ ]:
context_size = 4
sample = token_ids[50:]
for end in range(1, context_size + 1):
    context = sample[:end]
    target = sample[end]
    print(tokenizer.decode(context), '->', tokenizer.decode([target]))

In [ ]:
class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        token_ids = tokenizer.encode(text)
        self.input_ids = []
        self.target_ids = []
        for start in range(0, len(token_ids) - max_length, stride):
            inputs = token_ids[start:start + max_length]
            targets = token_ids[start + 1:start + max_length + 1]
            self.input_ids.append(torch.tensor(inputs))
            self.target_ids.append(torch.tensor(targets))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]


dataset = GPTDataset(raw_text, tokenizer, max_length=4, stride=4)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False, drop_last=True)
inputs, targets = next(iter(dataloader))
print('inputs:', inputs.shape)
print(inputs)
print('targets:', targets.shape)
print(targets)